<a href="https://colab.research.google.com/github/ibarr123/BUS1182026/blob/main/capstone_ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **  MULTI-AGENT AI SYSTEM FOR IT SUPPORT**

In [ ]:
#SETUP LIBRARIES

!pip install -q google-generativeai

from google.colab import userdata
import google.generativeai as genai

api_key = userdata.get('Capstone')

genai.configure(api_key=api_key)

model = genai.GenerativeModel("gemini-2.5-flash")

def ask_gemini(prompt):
    response = model.generate_content(prompt)
    return response.text

In [ ]:
#DATA

In [ ]:
#Agent 1: Intake Agent. This agent is in charge of reading the user request and classifying it into categories

import json

VALID_CATEGORIES = ["password", "vpn", "software", "hardware", "other"]

def intake_agent(user_input):
    text = user_input.lower()

    if "password" in text or "reset" in text or "login" in text:
        return "password"

    elif "vpn" in text or "network" in text:
        return "vpn"

    elif "install" in text or "software" in text or "app" in text:
        return "software"

    elif "laptop" in text or "hardware" in text or "overheat" in text:
        return "hardware"

    else:
        return "other"
    return category

In [ ]:
#TEST for intake agent
print("Password Test:", intake_agent("I forgot my password"))
print("VPN Test:", intake_agent("My VPN is not connecting"))
print("Software Test:", intake_agent("Zoom won’t install"))

Password Test: password
VPN Test: vpn
Software Test: software


In [ ]:
#Agent 2: Knowledge Agent (RAG). This agent retrieves answers from knowledge document and returns answers using RAG



!pip install -q sentence-transformers faiss-cpu

from sentence_transformers import SentenceTransformer
import faiss
import numpy as np


if "it_support_docs" not in globals():
    it_support_docs = [
        "Password resets: Users can reset their password through the IT portal by clicking 'Forgot Password' and verifying their identity with MFA.",
        "Account lockouts: If a user enters the wrong password too many times, the account may be locked for 15 minutes or require IT admin assistance.",
        "VPN issues: To connect to the company VPN, install the approved VPN client, verify internet access, and use your company credentials plus MFA.",
        "Software installation: Approved software can be installed through the company software center. Admin privileges may be required for restricted applications.",
        "Outlook troubleshooting: If Outlook will not open, restart the device, check for Office updates, and try opening Outlook in safe mode.",
        "Hardware issues: If a laptop overheats, check for blocked vents, close unused applications, and restart the machine. Persistent overheating should be escalated.",
        "Ticket triage: High-priority tickets include system outages, security incidents, and company-wide disruptions. Low-priority tickets include routine software requests.",
        "New user onboarding: New employees typically need account creation, email setup, VPN access, required software, and device provisioning."
    ]

# Load embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Create embeddings for docs
doc_embeddings = embedding_model.encode(it_support_docs, convert_to_numpy=True)
doc_embeddings = np.array(doc_embeddings).astype("float32")

# Build FAISS index
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(doc_embeddings)

def retrieve_docs(query, k=2):
    query_embedding = embedding_model.encode([query], convert_to_numpy=True)
    query_embedding = np.array(query_embedding).astype("float32")

    distances, indices = index.search(query_embedding, k)

    results = []
    for i in indices[0]:
        if 0 <= i < len(it_support_docs):
            results.append(it_support_docs[i])

    return results

# Agent 2: Knowledge Agent (RAG)

def knowledge_agent(user_input):
    retrieved_docs = retrieve_docs(user_input, k=2)

    if not retrieved_docs:
        return "I could not find enough information in the knowledge base."

    context = "\n".join(retrieved_docs)

    return f"""Knowledge Agent Response:
Based on the knowledge base, here is the relevant information:

{context}
"""

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
#TEST FOR KNOWLEDGE AGENT
print(knowledge_agent("How do I reset my password?"))

Knowledge Agent Response:
Based on the knowledge base, here is the relevant information:

Password resets: Users can reset their password through the IT portal by clicking 'Forgot Password' and verifying their identity with MFA.
Account lockouts: If a user enters the wrong password too many times, the account may be locked for 15 minutes or require IT admin assistance.



In [ ]:
#Agent 3: WorkFlow Agent (Action Taker). This agent simulates IT actions.
# Agent 3: Workflow Agent

def workflow_agent(user_input, category):
    if category == "password":
        return (
            "Workflow Agent Response:\n"
            "A password reset request has been initiated. "
            "Please use the IT portal 'Forgot Password' option and complete MFA verification."
        )

    elif category == "software":
        return (
            "Workflow Agent Response:\n"
            "A software support workflow has been started. "
            "Please check the company software center first. If the software requires admin approval, an IT ticket should be created."
        )

    elif category == "hardware":
        return (
            "Workflow Agent Response:\n"
            "A hardware diagnostic workflow has been started. "
            "Please restart the device, check power and connections, and note any unusual behavior for IT support."
        )

    elif category == "vpn":
        return (
            "Workflow Agent Response:\n"
            "A VPN troubleshooting workflow has been started. "
            "Please verify internet access, confirm the VPN client is installed, and retry with MFA."
        )

    else:
        return (
            "Workflow Agent Response:\n"
            "No direct workflow is available for this issue. Escalation may be required."
        )

In [ ]:
#TEST FOR WORKFLOW AGENT
print(workflow_agent("I forgot my password", "password"))
print(workflow_agent("Zoom won't install", "software"))
print(workflow_agent("My laptop is overheating", "hardware"))

Workflow Agent Response:
A password reset request has been initiated. Please use the IT portal 'Forgot Password' option and complete MFA verification.
Workflow Agent Response:
A software support workflow has been started. Please check the company software center first. If the software requires admin approval, an IT ticket should be created.
Workflow Agent Response:
A hardware diagnostic workflow has been started. Please restart the device, check power and connections, and note any unusual behavior for IT support.


In [ ]:
# Agent 4: Escalation Agent

ticket_counter = 1001

def escalation_agent(user_input):
    global ticket_counter
    ticket_id = ticket_counter
    ticket_counter += 1

    return (
        f"Escalation Agent Response:\n"
        f"This issue requires human IT support. Ticket #{ticket_id} has been created. "
        f"The request submitted was: '{user_input}'."
    )

In [ ]:
#Testing Escalation Agent
print(escalation_agent("My system has multiple issues and nothing is working"))

Escalation Agent Response:
This issue requires human IT support. Ticket #1002 has been created. The request submitted was: 'My system has multiple issues and nothing is working'.


In [ ]:
#This is what we call the "Orchestrator". It is in charge of making sure all agents work accorindgly with each other, like an orchestra, not each independently.
# Agent 5: Orchestrator

# Agent 5: Orchestrator

# Agent 5: Orchestrator

def orchestrator(user_input):
    category = intake_agent(user_input)

    print(f"Intake Agent classified this request as: {category}")

    if category in ["password", "vpn", "software", "hardware"]:
        knowledge_response = knowledge_agent(user_input)
        workflow_response = workflow_agent(user_input, category)

        return (
            f"\n--- FINAL SYSTEM RESPONSE ---\n"
            f"{knowledge_response}\n\n"
            f"{workflow_response}"
        )

    else:
        escalation_response = escalation_agent(user_input)

        return (
            f"\n--- FINAL SYSTEM RESPONSE ---\n"
            f"{escalation_response}"
        )

In [ ]:
#Testing Orchestrator
print(orchestrator("I forgot my password"))
print()
print(orchestrator("My VPN is not connecting"))
print()
print(orchestrator("Zoom won't install on my laptop"))
print()
print(orchestrator("My whole system is broken and I need urgent help"))

Intake Agent classified this request as: password

--- FINAL SYSTEM RESPONSE ---
Knowledge Agent Response:
Based on the knowledge base, here is the relevant information:

Password resets: Users can reset their password through the IT portal by clicking 'Forgot Password' and verifying their identity with MFA.
Account lockouts: If a user enters the wrong password too many times, the account may be locked for 15 minutes or require IT admin assistance.


Workflow Agent Response:
A password reset request has been initiated. Please use the IT portal 'Forgot Password' option and complete MFA verification.

Intake Agent classified this request as: vpn

--- FINAL SYSTEM RESPONSE ---
Knowledge Agent Response:
Based on the knowledge base, here is the relevant information:

VPN issues: To connect to the company VPN, install the approved VPN client, verify internet access, and use your company credentials plus MFA.
Outlook troubleshooting: If Outlook will not open, restart the device, check for Off